# Inline CodeMirror 6 Code Editor (`craftC2`)

> Inline CM6 editor for Solveit code cells, sharing CM6 core loading, syntax highlight styles, language loaders, find-box sync, and selection helpers via `window.__craftCMCore`.

In [ ]:
#| export_ft CRAFT.js
#| eval: false
TEMPLATES = dict(
    c2_tooltip_portal=Div(id='craft-c2-tooltips', klass='craft-c2-tooltips'),
    c2_ai_form=Form(
        craft_input(name='query', autocomplete='off', cls='craft-c2-ai-input',
                    placeholder='Enter your query... (Enter to apply, Esc to cancel)'),
        klass='craft-c2-ai-form'),
    c2_spinner=Div(klass='craft-c2-spinner', data_uk_spinner='ratio: 0.5'),
    c2_sighelp=Div(
        Div(
            Span(klass='craft-c2-sig-label'),
            Span(Button('▲', type='button', data_dir='-1'), Span(klass='craft-c2-sig-count'), Button('▼', type='button', data_dir='1'),
                 klass='craft-c2-sig-nav', hidden=True),
            klass='craft-c2-sig-head'),
        Div(klass='craft-c2-sig-meta', hidden=True),
        Pre(klass='craft-c2-sig-doc', hidden=True),
        klass='craft-c2-sighelp'),
)

## Preferences, Language Detection & Static Highlighting

Reads server/browser preferences, builds CM6 extras (ghost text, signature help, inline AI edit panel, indent guides, rainbow brackets), detects `%%magic` cell languages, and statically highlights inactive code cells.

In [ ]:
#| export_js_iife CRAFT.js craftC2
#| eval: false
window.craftC2?.disable();
  const CORE = window.__craftCMCore;
  if (!CORE.features.init('code-editor', 'Inline Code Editor (C2)', 'CodeMirror 6 inline editing for code cells')) return;
  const {SOLVEIT, mountTpl, qsa, on, store, patch, isMac, normId, LANG_ALIASES, langCache, isDark, buildStyles} = CORE;
  const staticHlText = new WeakMap(), smartSelHist = new WeakMap();
  const ghostTimers = new Map(), ghostAborts = new Map(), sigAborts = new Map();
  const getTooltipPortal = () => {
    let p = document.getElementById('craft-c2-tooltips');
    if (!p) { p = mountTpl('c2_tooltip_portal'); document.body.append(p); }
    return p;
  };

  const getServerPrefs = () => ({enterComp: false, autoClose: false});

  let cmCache = null;

  const kernelLang = () => (document.getElementById('kid')?.dataset.language || 'python').toLowerCase();
  const fimEnabled = () => kernelLang() === 'python';
  const dialogMode = () => document.querySelector('[name=mode]')?.value || '';
  const notifyErr = msg => CORE.notifyErr(msg, 'craftC2');
  const loadCM = CORE.loadCoreCM;
  loadCM().then(cm => { cmCache = cm; reconcileCells(CORE.allCells()); schedule(); }).catch(() => {});

  let cmExtras = null;
  const ensureCMExtras = cm => {
    if (!cm.ghLightStyle) buildStyles(cm);
    if (cmExtras) return cmExtras;
    const {state, view} = cm;
    const setGhost = state.StateEffect.define();
    const setSigHelp = state.StateEffect.define();
    const setAiEdit = state.StateEffect.define();

    class GhostWidget extends view.WidgetType {
      constructor(text) { super(); this.text = text; }
      eq(other) { return other.text === this.text; }
      toDOM() {
        return CORE.el('span', {cls: 'craft-c2-ghost'}, this.text);
      }
      ignoreEvent() { return true; }
    }

    const ghostField = state.StateField.define({
      create: () => null,
      update(val, tr) {
        for (const e of tr.effects) if (e.is(setGhost)) return e.value;
        if (!val) return null;
        if (tr.docChanged) {
          let next = null, count = 0;
          tr.changes.iterChanges((fromA, toA, _fromB, _toB, inserted) => {
            count++;
            if (count === 1 && fromA === toA && fromA === val.pos) {
              const ins = inserted.toString();
              if (ins && val.text.startsWith(ins) && ins.length < val.text.length) {
                next = {pos: val.pos + ins.length, text: val.text.slice(ins.length)};
              }
            }
          });
          return count === 1 ? next : null;
        }
        if (tr.selection) return null;
        return val;
      },
      provide: f => view.EditorView.decorations.from(f, g => (
        g ? view.Decoration.set([view.Decoration.widget({widget: new GhostWidget(g.text), side: 1}).range(g.pos)]) : view.Decoration.none
      )),
    });

    const buildSigTooltip = val => ({
      pos: val.pos,
      above: true,
      strictSide: false,
      arrow: false,
      create(edView) {
        const {el, mountTpl} = CORE;
        const {root: dom, label: sigEl, nav, count, meta, doc} = mountTpl('c2_sighelp', {refs: {
          label: '.craft-c2-sig-label', nav: '.craft-c2-sig-nav', count: '.craft-c2-sig-count',
          meta: '.craft-c2-sig-meta', doc: '.craft-c2-sig-doc',
        }});
        let latest = val;
        nav.addEventListener('mousedown', e => {
          const btn = e.target.closest('button[data-dir]');
          if (!btn || !latest?.values?.length) return;
          e.preventDefault();
          const n = latest.values.length;
          edView.dispatch({effects: setSigHelp.of({...latest, active: (latest.active + Number(btn.dataset.dir) + n) % n})});
        });
        const render = cur => {
          latest = cur;
          const s = cur.values[cur.active] || cur.values[0];
          if (!s) return;
          const descs = (s.params || []).map(p => typeof p === 'string' ? p : String(p.desc || p.name || '').replace(/^param /, ''));
          const fullLabel = String(s.label || ''), openIdx = fullLabel.indexOf('('), closeIdx = fullLabel.lastIndexOf(')');
          sigEl.replaceChildren(
            (openIdx >= 0 ? fullLabel.slice(0, openIdx) : fullLabel) + '(',
            ...descs.flatMap((d, i) => [i > 0 ? ', ' : null, i === s.idx ? el('strong', {cls: 'craft-c2-sig-active'}, d) : d]),
            closeIdx >= 0 ? fullLabel.slice(closeIdx) : ')');
          nav.hidden = cur.values.length <= 1;
          if (!nav.hidden) count.textContent = ` ${cur.active + 1}/${cur.values.length} `;
          meta.hidden = !(s.typ || s.mod);
          meta.replaceChildren(
            ...(s.typ ? [el('b', {}, 'Type: '), el('kbd', {}, s.typ), ' '] : []),
            ...(s.mod ? [el('b', {}, 'Module: '), el('kbd', {}, s.mod)] : []));
          doc.hidden = !s.doc;
          doc.textContent = s.doc || '';
        };
        render(val);
        return {
          dom,
          update(u) {
            const next = u.state.field(sigHelpField, false);
            if (next) render(next);
          },
        };
      },
    });

    const sigHelpField = state.StateField.define({
      create: () => null,
      update(val, tr) {
        for (const e of tr.effects) if (e.is(setSigHelp)) return e.value;
        if (tr.docChanged || tr.selection) return null;
        return val;
      },
      provide: f => view.showTooltip.from(f, val => val ? buildSigTooltip(val) : null),
    });

    class AiEditWidget extends view.WidgetType {
      constructor(spec) { super(); this.spec = spec; }
      eq(other) { return other.spec === this.spec; }
      toDOM() {
        const form = CORE.mountTpl('c2_ai_form');
        const input = form.querySelector('input');
        form.addEventListener('submit', e => {
          e.preventDefault();
          e.stopPropagation();
          this.spec.onSubmit(input.value);
        });
        input.addEventListener('keydown', e => {
          e.stopPropagation();
          if (e.key === 'Escape') {
            e.preventDefault();
            this.spec.onCancel();
          }
        });
        requestAnimationFrame(() => input.focus());
        return form;
      }
      ignoreEvent() { return true; }
    }

    const aiEditField = state.StateField.define({
      create: () => null,
      update(val, tr) {
        for (const e of tr.effects) if (e.is(setAiEdit)) return e.value;
        return val;
      },
      provide: f => view.EditorView.decorations.from(f, spec => (
        spec ? view.Decoration.set([view.Decoration.widget({widget: new AiEditWidget(spec), block: true, side: -1}).range(spec.lineFrom)]) : view.Decoration.none
      )),
    });

    const t = cm.hl.tags;
    const staticHighlighter = cm.hl.tagHighlighter([
      {tag: [t.comment, t.lineComment, t.blockComment, t.docComment], class: 'color:light-dark(#6a737d,#6a737d)'},
      {tag: [t.keyword, t.controlKeyword, t.operatorKeyword, t.definitionKeyword, t.moduleKeyword, t.operator, t.compareOperator, t.logicOperator, t.arithmeticOperator, t.bitwiseOperator, t.updateOperator], class: 'color:light-dark(#d73a49,#f97583)'},
      {tag: [t.string, t.special(t.string), t.docString], class: 'color:light-dark(#032f62,#9ecbff)'},
      {tag: [t.regexp], class: 'color:light-dark(#032f62,#dbedff)'},
      {tag: [t.self, t.escape, t.number, t.integer, t.float, t.bool, t.null, t.atom, t.constant(t.variableName), t.propertyName, t.attributeName, t.standard(t.variableName), t.special(t.variableName), t.link, t.url], class: 'color:light-dark(#005cc5,#79b8ff)'},
      {tag: [t.function(t.variableName), t.function(t.propertyName), t.definition(t.function(t.variableName)), t.className, t.definition(t.className), t.typeName, t.meta, t.annotation], class: 'color:light-dark(#6f42c1,#b392f0)'},
      {tag: [t.tagName], class: 'color:light-dark(#22863a,#85e89d)'},
      {tag: [t.heading], class: 'color:light-dark(#005cc5,#79b8ff);font-weight:700'},
      {tag: [t.strong], class: 'color:light-dark(#24292e,#e1e4e8);font-weight:700'},
      {tag: [t.emphasis], class: 'color:light-dark(#24292e,#e1e4e8);font-style:italic'},
      {tag: [t.variableName, t.definition(t.variableName), t.punctuation, t.bracket, t.paren, t.squareBracket, t.brace], class: 'color:light-dark(#24292e,#e1e4e8)'},
      {tag: [t.invalid], class: 'color:light-dark(#cb2431,#fdaeb7);font-style:italic'},
    ]);

    const igDeco = view.Decoration.mark({class: 'craft-c2-ig'});
    const buildIndentGuides = edView => {
      const builder = new state.RangeSetBuilder();
      const doc = edView.state.doc;
      const seen = new Set();
      for (const {from, to} of edView.visibleRanges) {
        let pos = from;
        while (pos <= to) {
          const line = doc.lineAt(pos);
          if (!seen.has(line.number)) {
            seen.add(line.number);
            const m = line.text.match(/^ +/);
            if (m && m[0].length < line.text.length) {
              const spaces = m[0].length;
              for (let col = 0; col + 4 <= spaces; col += 4) {
                builder.add(line.from + col, line.from + col + 1, igDeco);
              }
            }
          }
          if (line.to >= doc.length) break;
          pos = line.to + 1;
        }
      }
      return builder.finish();
    };
    const indentGuidesExt = view.ViewPlugin.fromClass(class {
      constructor(edView) { this.decorations = buildIndentGuides(edView); }
      update(u) { if (u.docChanged || u.viewportChanged) this.decorations = buildIndentGuides(u.view); }
    }, {decorations: p => p.decorations});

    const brDecos = [
      view.Decoration.mark({class: 'craft-c2-br-0'}),
      view.Decoration.mark({class: 'craft-c2-br-1'}),
      view.Decoration.mark({class: 'craft-c2-br-2'}),
    ];
    const brErrDeco = view.Decoration.mark({class: 'craft-c2-br-err'});
    const BR_OPEN = {'(': ')', '[': ']', '{': '}'};
    const BR_CLOSE = new Set([')', ']', '}']);
    const SKIP_NODES = new Set(['Comment', 'LineComment', 'BlockComment', 'String', 'FormatString', 'TemplateString', 'RegExp', 'RawString']);
    const buildRainbowBrackets = edView => {
      const st = edView.state;
      const doc = st.doc;
      if (doc.length > 50_000) return view.Decoration.none;
      const visible = edView.visibleRanges;
      if (!visible.length) return view.Decoration.none;
      const maxTo = visible[visible.length - 1].to;
      const text = doc.sliceString(0, maxTo);
      const tree = cm.lang.syntaxTree(st);
      const builder = new state.RangeSetBuilder();
      const stack = [];
      let vIdx = 0;
      for (let i = 0; i < text.length; i++) {
        const ch = text[i];
        const isOpen = Boolean(BR_OPEN[ch]);
        if (!isOpen && !BR_CLOSE.has(ch)) continue;
        let skip = false;
        for (let n = tree.resolveInner(i, 1); n; n = n.parent) {
          if (SKIP_NODES.has(n.name)) { skip = true; break; }
        }
        if (skip) continue;
        while (vIdx < visible.length && visible[vIdx].to <= i) vIdx++;
        const inVis = vIdx < visible.length && i >= visible[vIdx].from && i < visible[vIdx].to;
        if (isOpen) {
          const depth = stack.length;
          stack.push({close: BR_OPEN[ch], depth});
          if (inVis) builder.add(i, i + 1, brDecos[depth % 3]);
        } else {
          let matched = null;
          for (let j = stack.length - 1; j >= 0; j--) {
            if (stack[j].close === ch) { matched = stack.splice(j)[0]; break; }
          }
          if (inVis) builder.add(i, i + 1, matched ? brDecos[matched.depth % 3] : brErrDeco);
        }
      }
      return builder.finish();
    };
    const rainbowBracketsExt = view.ViewPlugin.fromClass(class {
      constructor(edView) { this.decorations = buildRainbowBrackets(edView); }
      update(u) { if (u.docChanged || u.viewportChanged || cm.lang.syntaxTree(u.startState) !== cm.lang.syntaxTree(u.state)) this.decorations = buildRainbowBrackets(u.view); }
    }, {decorations: p => p.decorations});

    cmExtras = {setGhost, ghostField, setSigHelp, sigHelpField, setAiEdit, aiEditField, staticHighlighter, indentGuidesExt, rainbowBracketsExt};
    return cmExtras;
  };

## Language Detection, Static Highlighting & Smart Selection
Detects cell languages (including `%%magic` overrides), lazily loads CM6 language extensions, statically highlights unmounted code cells, and implements AST-based smart selection expand/shrink.

In [ ]:
#| export_js_iife CRAFT.js craftC2
#| eval: false
  const themeExtensions = cm => {
    ensureCMExtras(cm);
    return [
      cm.lang.syntaxHighlighting(isDark() ? cm.ghDarkStyle : cm.ghLightStyle),
      isDark() ? cm.darkTheme : cm.lightTheme,
    ];
  };

  const detectLangKey = text => {
    const kl = kernelLang();
    if (kl === 'python') {
      const m = String(text || '').match(/^%%(\w+)/)?.[1]?.toLowerCase();
      if (m) return LANG_ALIASES[m] || m;
    }
    return LANG_ALIASES[kl] || kl;
  };

    async function resolveLangExt(cm, langKey) {
    try { return await CORE.loadLang(cm, langKey); }
    catch (err) { console.warn('craftC2: language fallback to plaintext for', langKey, err); return []; }
  }

  async function syncCellLanguage(id, v) {
    const rec = views.get(id);
    if (!rec || !cmCache) return;
    const want = detectLangKey(v.state.doc.toString());
    if (rec.langKey === want) return;
    rec.langKey = want;
    const ext = await resolveLangExt(cmCache, want);
    const cur = views.get(id);
    if (!cur || cur !== rec || cur.langKey !== want) return;
    cur.view.dispatch({effects: cur.langComp.reconfigure(ext)});
  }

  function renderStaticHighlight(codeEl, text, tree, langKey) {
    if (!cmCache || !codeEl || !tree) return;
    const {staticHighlighter} = ensureCMExtras(cmCache);
    const frag = document.createDocumentFragment();
    let pos = 0;
    cmCache.hl.highlightTree(tree, staticHighlighter, (from, to, cls) => {
      if (from > pos) frag.append(document.createTextNode(text.slice(pos, from)));
      if (cls) {
        frag.append(CORE.el('span', {style: cls.trim().replace(/\s+/g, ';')}, text.slice(from, to)));
      } else {
        frag.append(document.createTextNode(text.slice(from, to)));
      }
      pos = to;
    });
    if (pos < text.length) frag.append(document.createTextNode(text.slice(pos)));
    codeEl.replaceChildren(frag);
    codeEl.dataset.highlighted = 'true';
    staticHlText.set(codeEl, {text, langKey});
  }

  function ensureStaticHighlighted(card) {
    if (!cmCache || !card) return;
    const codeEl = card.querySelector(`${SOLVEIT.sel.msgBody} ${SOLVEIT.sel.msgContent} pre code`);
    if (!codeEl) return;
    const text = codeEl.textContent || '';
    if (!text.trim()) return;
    const langKey = detectLangKey(text);
    const prev = staticHlText.get(codeEl);
    if (prev && prev.text === text && prev.langKey === langKey && codeEl.querySelector('span[style]')) return;
    if (langKey === 'python') {
      const parser = cmCache.python.pythonLanguage?.parser || cmCache.python.python().language?.parser;
      if (parser) renderStaticHighlight(codeEl, text, parser.parse(text), langKey);
      return;
    }
    const cacheKey = `${cmCache._source || 'cdn'}:${langKey}`;
    const cached = langCache.get(cacheKey);
    const parser = cached?.language?.parser || cached?.parser;
    if (parser) {
      renderStaticHighlight(codeEl, text, parser.parse(text), langKey);
      return;
    }
    resolveLangExt(cmCache, langKey).then(ext => {
      if (ed.destroyed || !codeEl.isConnected) return;
      const curCard = codeEl.closest(SOLVEIT.sel.cardInType);
      if (curCard?.classList.contains('craft-c2-mounted')) return;
      if ((codeEl.textContent || '') !== text) return;
      const p = ext?.language?.parser || ext?.parser;
      if (p) renderStaticHighlight(codeEl, text, p.parse(text), langKey);
    });
  }

    const syncFindBoxToView = v => CORE.syncFindBoxToView(v, cmCache);

  function smartExpandSelection(v) {
    if (!cmCache) return false;
    const st = v.state;
    const sel = st.selection.main;
    const tree = cmCache.lang.ensureSyntaxTree(st, st.doc.length, 150) || cmCache.lang.syntaxTree(st);
    let node = tree.resolveInner(sel.from, 1);
    if (!node || (node.from === sel.from && node.to === sel.to && sel.from > 0)) {
      const alt = tree.resolveInner(sel.from, -1);
      if (alt) node = alt;
    }
    while (node && !(node.from <= sel.from && node.to >= sel.to && (node.from < sel.from || node.to > sel.to))) {
      node = node.parent;
    }
    if (!node) return false;
    let hist = smartSelHist.get(v);
    const cur = {from: sel.from, to: sel.to};
    if (!hist || hist.doc !== st.doc || !hist.last || hist.last.from !== cur.from || hist.last.to !== cur.to) {
      hist = {doc: st.doc, stack: [], last: null};
    }
    hist.stack.push(cur);
    hist.last = {from: node.from, to: node.to};
    smartSelHist.set(v, hist);
    v.dispatch({selection: {anchor: node.from, head: node.to}, scrollIntoView: true});
    return true;
  }

  function smartShrinkSelection(v) {
    const hist = smartSelHist.get(v);
    const sel = v.state.selection.main;
    if (!hist || hist.doc !== v.state.doc || !hist.stack.length || !hist.last || hist.last.from !== sel.from || hist.last.to !== sel.to) {
      return false;
    }
    const prev = hist.stack.pop();
    hist.last = hist.stack.length ? prev : null;
    v.dispatch({selection: {anchor: prev.from, head: prev.to}, scrollIntoView: true});
    return true;
  }

  async function switchCellType(id, v, mtyp) {
    if (mtyp === 'code') return;
    await ed.switchCellType(id, v, mtyp, {
      onMatch: window.craftMd ? (cid, selSpec) => window.craftMd.edit(cid, selSpec) : null,
    });
  }

## Solveit Lifecycle, `createInlineEditor` & Auto-Edit Handoff

Wires `window.__craftCMCore.createInlineEditor` for code cells, tracks line-number toggles and tracebacks, and coordinates Monaco handover and auto-edit claiming after `a`/`b`/`Alt+Enter` cell insertions.

In [ ]:
#| export_js_iife CRAFT.js craftC2
#| eval: false
const jumpToLastCell = id => CORE.jumpToLastCell(() => exitCell(id));

  const getCellEl = id => document.getElementById('_' + id);
  const getCardIn = el => el?.querySelector(SOLVEIT.sel.codeCard);
  const getCellText = el => el?.querySelector('form [name="content"]')?.value ?? '';
  const hasDiff = el => !!(el && (el.dataset.diff || el.querySelector('[data-diff]')));
  let nonCodeInsertion = null;
  const isNonCodeInsertion = id => nonCodeInsertion && !nonCodeInsertion.has(id);
  const isC2El = el => !!el && !isNonCodeInsertion(el.id) && !!getCardIn(el) && !hasDiff(el);
  const typingBlocked = () => CORE.typingBlocked();

  const ed = CORE.createInlineEditor({
    tag: 'craftC2',
    mountedCls: 'craft-c2-mounted',
    getCard: el => getCardIn(el),
    getText: el => getCellText(el),
    getInput: el => el?.querySelector('form [name="content"]'),
    applySaveState: (id, st) => {
      const el = getCellEl(id);
      if (!el) return;
      if (st) el.dataset.c2Save = st;
      else delete el.dataset.c2Save;
    },
    setEditing: (_id, on, el) => {
      if (!el) return;
      if (on) el.dataset.c2Editing = 'true';
      else delete el.dataset.c2Editing;
    },
    beforeUnmount: (id, rec) => {
      cancelGhost(id);
      const ab = sigAborts.get(id);
      if (ab) { ab.abort(); sigAborts.delete(id); }
      if (!rec) return;
      const docText = rec.view.state.doc.toString();
      const codeEl = getCardIn(getCellEl(id))?.querySelector(`${SOLVEIT.sel.msgBody} ${SOLVEIT.sel.msgContent} pre code`);
      if (codeEl) {
        if (cmCache && docText.trim()) {
          const tree = cmCache.lang.ensureSyntaxTree(rec.view.state, docText.length, 200) || cmCache.lang.syntaxTree(rec.view.state);
          renderStaticHighlight(codeEl, docText, tree, rec.langKey || detectLangKey(docText));
        } else {
          codeEl.textContent = docText;
          staticHlText.delete(codeEl);
        }
      }
    },
    beforeBlur: id => cancelGhost(id),
    beforeExit: id => cancelGhost(id),
    beforeRun: id => cancelGhost(id),
    beforeSplit: (id, v) => cancelGhost(id, v),
    onRekey: oldId => {
      cancelGhost(oldId);
      const ab = sigAborts.get(oldId);
      if (ab) { ab.abort(); sigAborts.delete(oldId); }
    },
    keepMounted: (_id, rec) => Boolean(cmCache && hasFolds(cmCache, rec.view.state)),
    isViewValid: (_id, _rec, el) => !hasDiff(el) && isC2El(el),
    onSync: () => {
      reconcileAutoEdits();
      driveAutoEdit();
    },
    onBeforeRequest: (ctx, pathname, body) => {
      if (pathname === SOLVEIT.api.addAbove || pathname === SOLVEIT.api.addBelow) {
        if (nonCodeInsertion || CORE.optimistic?.isSuppressingAutoEdit?.()) return;
        if (!pendingAutoEdits.some(r => r.forceCode && r.expires > Date.now())) queueAutoEdit({forceCode: false});
        return;
      }
      if (pathname === SOLVEIT.api.addRunq) {
        if (body.get('cmd') === 'alt-run') queueAutoEdit({forceCode: false, ttl: 30000});
        return;
      }
    },
    openPrimaryOnEnter: (el, run) => {
      if (!isC2El(el)) return false;
      const card = getCardIn(el);
      if (!card || card.classList.contains('collapsed')) return false;
      if (run) openCell(el.id.replace(/^_/, ''));
      return true;
    },
    onEditClick: (edit, e) => {
      const cell = edit.closest(SOLVEIT.sel.cell), cellId = cell?.id?.replace(/^_/, ''), codeCard = getCardIn(cell);
      if (!cellId || !codeCard || hasDiff(cell) || isNonCodeInsertion(cell.id) || codeCard.classList.contains('collapsed')) return false;
      e.preventDefault(); e.stopImmediatePropagation();
      openCell(cellId);
      return true;
    },
    getThemeExts: () => (cmCache ? themeExtensions(cmCache) : null),
    beforeSwitch: (id, v) => cancelGhost(id, v),
    failRenameMsg: 'Could not save code edit before reopening renamed dialog.',
    domAttrs: ['value', 'data-diff'],
    inDialog: false,
    pointerEdit: {
      ignoreSel: 'button, a, .craft-c2-host',
      resolveTarget: e => {
        const content = e.target.closest?.(`${SOLVEIT.sel.cell} ${SOLVEIT.sel.codeCard} ${SOLVEIT.sel.msgContent}`);
        if (!content) return null;
        const el = content.closest(SOLVEIT.sel.cell);
        if (!isC2El(el)) return null;
        return {id: normId(el.id), codeEl: content.querySelector('pre code')};
      },
      onOpen: (p, selSpec) => openCell(p.id, selSpec),
    },
  });
  const {states, views, pendingSaves, dirtyIds, failedSaves, syncSaveState, saveCell, unmountCell, exitCell, saveAndRun, sync, schedule} = ed;
  function reconcileCells(cells) {
    for (const el of cells) {
      const card = el.querySelector(SOLVEIT.sel.cardIn);
      if (!card) continue;
      syncSaveState(normId(el.id));
      if (card.dataset.mtype === 'code' && !card.classList.contains('craft-c2-mounted') && !card.closest('[data-streaming]')) {
        ensureStaticHighlighted(card);
      }
    }
  }
  const unsubCells = CORE.onCells(reconcileCells, {attrs: ['data-mtype', 'data-streaming'], initial: true});
  const splitCell = (id, v) => ed.splitCell(id, v, 'code');

  const pendingAutoEdits = [];
  let autoEditGuard = null;

  const clearAutoEditGuard = () => { autoEditGuard = null; };

  const queueAutoEdit = ({forceCode = false, ttl = 6000} = {}) => {
    clearAutoEditGuard();
    const existing = new Set(qsa(SOLVEIT.sel.cellWithId).map(el => el.id));
    pendingAutoEdits.push({existing, forceCode, expires: Date.now() + ttl});
  };

  const claimNewCell = (nid, {forceCode = false} = {}) => {
    clearAutoEditGuard();
    autoEditGuard = {id: nid, forceCode, wantOpen: true, converted: false, expires: Date.now() + 4000};
    driveAutoEdit();
  };

  const driveAutoEdit = () => {
    if (!autoEditGuard) return;
    const g = autoEditGuard;
    if (Date.now() > g.expires) { clearAutoEditGuard(); return; }
    const el = getCellEl(g.id);
    const card = el?.querySelector(SOLVEIT.sel.cardIn);
    const isCode = card?.dataset.mtype === 'code';
    if (el && card && !isCode) {
      if (g.forceCode) {
        if (!g.converted) {
          g.converted = true;
          CORE.htmxPost(SOLVEIT.api.setMtyp, {id_: '_' + g.id, ids: '_' + g.id, mtyp: 'code'})
            .catch(err => console.error('craftC2: set_mtyp failed', err));
        }
        return;
      }
      clearAutoEditGuard();
      return;
    }
    if (el && isCode && !hasDiff(el) && g.wantOpen) {
      g.wantOpen = false;
      if (el.dataset.editing) delete el.dataset.editing;
      el.scrollIntoView({block: 'nearest'});
      openCell(g.id, {anchor: 0, head: 0});
    }
  };

  const reconcileAutoEdits = () => {
    if (!pendingAutoEdits.length) return;
    const now = Date.now();
    for (let i = pendingAutoEdits.length - 1; i >= 0; i--) {
      if (pendingAutoEdits[i].expires < now) pendingAutoEdits.splice(i, 1);
    }
    if (!pendingAutoEdits.length) return;
    const cells = qsa(SOLVEIT.sel.cellWithId);
    for (let i = 0; i < pendingAutoEdits.length; i++) {
      const req = pendingAutoEdits[i];
      const added = cells.filter(el => !isNonCodeInsertion(el.id) && !req.existing.has(el.id) && (req.forceCode ? Boolean(el.querySelector(SOLVEIT.sel.cardIn)) : Boolean(getCardIn(el))));
      if (!added.length) continue;
      pendingAutoEdits.splice(i, 1);
      claimNewCell(added[0].id.replace(/^_/, ''), req);
      break;
    }
  };

  async function newCodeCell() {
    if (ed.destroyed) return false;
    if (ed.activeKey) await exitCell(ed.activeKey);
    const kid = CORE.kid();
    if (!kid) return false;
    const allCells = qsa(SOLVEIT.sel.cellWithId);
    const last = allCells[allCells.length - 1];
    queueAutoEdit({forceCode: true});
    try {
      if (last) {
        const lastId = last.id.replace(/^_/, '');
        const after = window.sectionTail ? window.sectionTail(last) : last.id;
        await CORE.htmxPost(SOLVEIT.api.addBelow, {id_: lastId, after});
      } else {
        await CORE.htmxPost(SOLVEIT.api.upsertMsg, {id_: '', msg_type: 'code', is_input: '1', content: ' '});
      }
      schedule();
      const existing = new Set(allCells.map(el => el.id));
      const added = await CORE.waitForHtmxSettle(() =>
        CORE.allCells().find(el => !existing.has(el.id) && isC2El(el) && el.querySelector('.craft-c2-host')) || null
      , 6000);
      const addedId = added ? added.id.slice(1) : null;
      if (addedId && !document.activeElement?.closest?.(`#_${addedId} .craft-c2-host`)) openCell(addedId, {anchor: 0, head: 0});
      return true;
    } catch (err) {
      pendingAutoEdits.pop();
      notifyErr('Could not create code cell.');
      console.error('craftC2: newCodeCell failed', err);
      return false;
    }
  }

  function stepBoundary(id, v, up) {
    if (!cmCache || !cmExtras) return false;
    const {sigHelpField, aiEditField} = cmExtras;
    return ed.stepBoundary(id, v, up, {
      tol: 2,
      canStep: view => !(cmCache.ac.completionStatus(view.state) || view.state.field(sigHelpField, false) || view.state.field(aiEditField, false)),
      beforeStep: (cid, view) => cancelGhost(cid, view),
    });
  }

## Kernel Completions, Ghost Text, Signature Help & Inline AI (`SuperComplete`)

Implements kernel-backed autocomplete (`kernelComplete`), local word fallback (`wordComplete`), FIM ghost-text scheduling/acceptance, signature help popups, and inline AI edits (`triggerSuperComplete`).

In [ ]:
#| export_js_iife CRAFT.js craftC2
#| eval: false
const hasFolds = (cm, state) => {
    let any = false;
    cm.lang.foldedRanges(state).between(0, state.doc.length, () => { any = true; return false; });
    return any;
  };

  const TRIGGER_CHARS = new Set(['.', '(', '[', '{', '@', ',', ' ', '+', '-', '/', '*', '%', '^']);
  const SURROUND_PAIRS = {'(': ')', '[': ']', '{': '}', "'": "'", '"': '"'};

  const inCommentOrString = (cm, st, pos) => {
    const name = cm.lang.syntaxTree(st).resolveInner(pos, -1)?.name || '';
    return name === 'Comment' || name === 'String' || name === 'FormatString';
  };

  const kindMap = {
    Function: 'function', Method: 'method', Class: 'class', Module: 'namespace',
    Variable: 'variable', Property: 'property', Keyword: 'keyword', Constant: 'constant',
    Field: 'property', Interface: 'interface', Enum: 'enum', File: 'text', Folder: 'text',
  };

  async function kernelComplete(context) {
    if (window.svCopts?.scon === false) return null;
    const kid = document.getElementById('kid')?.value;
    if (!kid) return null;
    if (!context.explicit && cmCache && inCommentOrString(cmCache, context.state, context.pos)) return null;
    const word = context.matchBefore(/[\w./]+/);
    const prevCh = context.pos > 0 ? context.state.doc.sliceString(context.pos - 1, context.pos) : '';
    if (!context.explicit && !word && !TRIGGER_CHARS.has(prevCh)) return null;
    const doc = context.state.doc;
    const code = doc.toString();
    if (!code) return null;
    const ctrl = new AbortController();
    let timer = null;
    context.addEventListener('abort', () => { clearTimeout(timer); ctrl.abort(); });
    await new Promise(resolve => { timer = setTimeout(resolve, 250); });
    if (context.aborted) return null;
    const line = doc.lineAt(context.pos);
    try {
      const {completions} = await CORE.fetchJson('/complete_', {
        body: {code, line_no: line.number, col_no: context.pos - line.from + 1, kid},
        signal: ctrl.signal,
      });
      if (!Array.isArray(completions) || !completions.length || context.aborted) return null;
      const clamp = n => Math.max(0, Math.min(doc.length, n));
      const toPos = (ln, col) => clamp(doc.line(Math.max(1, Math.min(doc.lines, ln))).from + Math.max(0, col - 1));
      let from = word ? word.from : context.pos, to = context.pos;
      const firstRange = completions[0]?.range;
      if (firstRange) {
        from = toPos(firstRange.startLineNumber, firstRange.startColumn);
        to = toPos(firstRange.endLineNumber, firstRange.endColumn);
      }
      const options = completions.map(c => {
        let apply = c.text;
        if (c.range) {
          const itemFrom = toPos(c.range.startLineNumber, c.range.startColumn);
          const itemTo = toPos(c.range.endLineNumber, c.range.endColumn);
          if (itemFrom !== from || itemTo !== to) {
            apply = (edView, _comp, _curFrom, curTo) => {
              const delta = curTo - to;
              const maxLen = edView.state.doc.length;
              const a = Math.max(0, Math.min(maxLen, itemFrom));
              const b = Math.max(a, Math.min(maxLen, itemTo + delta));
              edView.dispatch({changes: {from: a, to: b, insert: c.text}, selection: {anchor: a + c.text.length}});
            };
          }
        }
        return {
          label: c.text,
          apply,
          detail: c.detail || undefined,
          type: kindMap[c.kind] || 'variable',
          boost: -(Number(c.rank) || 0),
        };
      });
      return {from, to, options, validFor: /^\w*$/};
    } catch { return null; }
  }

  function wordComplete(context) {
    if (!context.explicit && cmCache && inCommentOrString(cmCache, context.state, context.pos)) return null;
    const word = context.matchBefore(context.explicit ? /[A-Za-z_]\w*/ : /[A-Za-z_]\w+/);
    if (!word) return null;
    const seen = new Set([word.text]);
    const options = [];
    const text = context.state.doc.toString();
    for (const m of text.matchAll(/\b[A-Za-z_]\w+\b/g)) {
      const w = m[0];
      if (seen.has(w)) continue;
      seen.add(w);
      options.push({label: w, type: 'text', boost: -100});
      if (options.length >= 200) break;
    }
    return options.length ? {from: word.from, to: word.to, options, validFor: /^\w*$/} : null;
  }

  function cancelGhost(id, v = null) {
    clearTimeout(ghostTimers.get(id));
    ghostTimers.delete(id);
    const ab = ghostAborts.get(id);
    if (ab) { ab.abort(); ghostAborts.delete(id); }
    const edView = v || views.get(id)?.view;
    if (edView && cmExtras && edView.state.field(cmExtras.ghostField, false)) {
      edView.dispatch({effects: cmExtras.setGhost.of(null)});
    }
  }

  async function requestGhost(id, v, explicit = false) {
    if (ed.destroyed || !cmCache || !fimEnabled()) return;
    const copts = window.svCopts;
    const kid = document.getElementById('kid')?.value;
    if (!kid || copts?.scon === false) return;
    if (!v.hasFocus || !v.state.selection.main.empty || v.state.readOnly) return;
    if (v.state.doc.length > 32_000) return;
    if (!explicit && dialogMode() === 'learning') return;
    if (cmCache.ac.selectedCompletionIndex(v.state) != null) return;
    const pos = v.state.selection.main.head;
    const doc = v.state.doc;
    const line = doc.lineAt(pos);
    const triggerChars = typeof copts?.triggerc === 'string' ? (() => { try { return JSON.parse(copts.triggerc); } catch { return []; } })() : copts?.triggerc;
    if (!explicit && !triggerChars?.some(c => line.text.endsWith(c))) return;
    const prevAb = ghostAborts.get(id);
    if (prevAb) prevAb.abort();
    const ctrl = new AbortController();
    ghostAborts.set(id, ctrl);
    const pfx = doc.sliceString(0, pos), sfx = doc.sliceString(pos);
    const prevLine = doc.line(Math.max(line.number - 1, 1)).text;
    const req_id = Math.random().toString(36).slice(2);
    const origText = doc.toString();
    try {
      const {completions} = await CORE.fetchJson('/ghost_', {
        body: {id_: id, mtype: 'code', output: false, pfx, sfx, req_id, kid},
        signal: ctrl.signal,
      });
      if (!Array.isArray(completions) || !completions.length || ctrl.signal.aborted) return;
      const cur = views.get(id);
      if (!cur || cur.view !== v || !v.hasFocus || v.state.doc.toString() !== origText || v.state.selection.main.head !== pos || !v.state.selection.main.empty) return;
      const first = completions[0].text;
      if (!first) return;
      if (/^\s*$/.test(first) && copts?.acceptichars && !sfx.length && !/^\s*$/.test(prevLine)) {
        v.dispatch({changes: {from: pos, to: pos, insert: first}, selection: {anchor: pos + first.length}});
        if (fimEnabled() && copts?.triggeronaccept && dialogMode() !== 'learning') {
          clearTimeout(ghostTimers.get(id));
          ghostTimers.set(id, setTimeout(() => {
            ghostTimers.delete(id);
            if (!ed.destroyed) requestGhost(id, v, false);
          }, 50));
        }
        return;
      }
      const {setGhost} = ensureCMExtras(cmCache);
      v.dispatch({effects: setGhost.of({pos, text: first})});
    } catch (err) {
      if (err?.name !== 'AbortError') console.error('craftC2: ghost error', err);
    } finally {
      if (ghostAborts.get(id) === ctrl) ghostAborts.delete(id);
    }
  }

  function scheduleGhost(id, v) {
    clearTimeout(ghostTimers.get(id));
    const prevAb = ghostAborts.get(id);
    if (prevAb) { prevAb.abort(); ghostAborts.delete(id); }
    if (!fimEnabled() || window.svCopts?.scon === false || dialogMode() === 'learning') return;
    ghostTimers.set(id, setTimeout(() => {
      ghostTimers.delete(id);
      const cur = views.get(id);
      if (cur && cur.view === v && v.hasFocus) requestGhost(id, v, false);
    }, 1500));
  }

  function acceptGhost(id, v) {
    if (!cmExtras || !v.state.selection.main.empty) return false;
    const {ghostField, setGhost} = cmExtras;
    const g = v.state.field(ghostField, false);
    if (!g) return false;
    v.dispatch({
      changes: {from: g.pos, to: g.pos, insert: g.text},
      selection: {anchor: g.pos + g.text.length},
      effects: setGhost.of(null),
    });
    if (fimEnabled() && window.svCopts?.triggeronaccept && dialogMode() !== 'learning') {
      clearTimeout(ghostTimers.get(id));
      ghostTimers.set(id, setTimeout(() => {
        ghostTimers.delete(id);
        const cur = views.get(id);
        if (!ed.destroyed && cur && cur.view === v && v.hasFocus) requestGhost(id, v, false);
      }, 50));
    }
    return true;
  }

  async function triggerSigHelp(id, v) {
    if (!cmCache || !fimEnabled() || !v.state.selection.main.empty) return false;
    const {sigHelpField, setSigHelp} = ensureCMExtras(cmCache);
    if (v.state.field(sigHelpField, false)) return false;
    const kid = document.getElementById('kid')?.value;
    if (!kid) return false;
    const prevAb = sigAborts.get(id);
    if (prevAb) prevAb.abort();
    const ctrl = new AbortController();
    sigAborts.set(id, ctrl);
    const pos = v.state.selection.main.head;
    const doc = v.state.doc;
    const line = doc.lineAt(pos);
    const origText = doc.toString();
    try {
      const {values} = await CORE.fetchJson('/sig_help_', {
        body: {code: origText, line_no: line.number, col_no: pos - line.from, kid},
        signal: ctrl.signal,
      });
      if (ctrl.signal.aborted) return false;
      let items = [], active = 0;
      if (Array.isArray(values)) {
        items = values.map(s => ({
          label: String(s.label || ''),
          params: (s.params || []).map(p => String(p.desc || p.name || '').replace(/^param /, '')),
          idx: s.idx ?? 0,
          typ: s.typ || '',
          mod: s.mod || '',
          doc: s.doc || '',
        }));
      } else if (values && Array.isArray(values.signatures)) {
        active = values.activeSignature || 0;
        items = values.signatures.map(s => ({
          label: String(s.label || ''),
          params: (s.parameters || []).map(p => (typeof p === 'string' ? p : String(p.label || ''))),
          idx: s.activeParameter ?? values.activeParameter ?? 0,
          typ: s.typ || '',
          mod: s.mod || '',
          doc: typeof s.documentation === 'string' ? s.documentation : (s.documentation?.value || ''),
        }));
      }
      if (!items.length) return false;
      const cur = views.get(id);
      if (!cur || cur.view !== v || v.state.doc.toString() !== origText || v.state.selection.main.head !== pos) return false;
      cmCache.ac.closeCompletion(v);
      v.dispatch({effects: setSigHelp.of({pos, values: items, active: Math.max(0, Math.min(items.length - 1, active))})});
      return true;
    } catch (err) {
      if (err?.name !== 'AbortError') console.error('craftC2: sig_help error', err);
      return false;
    } finally {
      if (sigAborts.get(id) === ctrl) sigAborts.delete(id);
    }
  }

  function setCellSpinner(rec, on) {
    if (!rec?.host) return;
    let sp = rec.host.querySelector('.craft-c2-spinner');
    if (on && !sp) {
      sp = CORE.mountTpl('c2_spinner');
      rec.host.append(sp);
    } else if (!on && sp) sp.remove();
  }

  async function runSuperComplete(id, v, hl, instr, from, to) {
    const rec = views.get(id);
    if (!rec || !cmCache || !fimEnabled()) return;
    const kid = document.getElementById('kid')?.value;
    if (!kid) return;
    v.dispatch({effects: rec.readOnlyComp.reconfigure(cmCache.state.EditorState.readOnly.of(true))});
    setCellSpinner(rec, true);
    try {
      const doc = v.state.doc;
      const pfx = doc.sliceString(0, from), sfx = doc.sliceString(to);
      const {completions} = await CORE.fetchJson(hl ? '/supere_' : '/superc_', {
        body: {id_: id, mtype: 'code', output: false, pfx, sfx, hl, instr, kid},
        timeoutMs: 60000,
      });
      const cur = views.get(id);
      if (cur && cur.view === v) {
        v.dispatch({effects: rec.readOnlyComp.reconfigure(cmCache.state.EditorState.readOnly.of(false))});
        if (Array.isArray(completions) && completions.length) {
          const txt = completions[0].text;
          v.dispatch({changes: {from, to, insert: txt}, selection: {anchor: from + txt.length}});
        }
        v.focus();
      }
    } catch (err) {
      const cur = views.get(id);
      if (cur && cur.view === v) {
        v.dispatch({effects: rec.readOnlyComp.reconfigure(cmCache.state.EditorState.readOnly.of(false))});
        v.focus();
      }
      notifyErr(err?.message || 'Completion request failed.');
    } finally {
      setCellSpinner(views.get(id), false);
    }
  }

  function triggerSuperComplete(id, v) {
    const rec = views.get(id);
    if (!rec || !cmCache || !fimEnabled() || v.state.readOnly) return false;
    cancelGhost(id, v);
    const {setAiEdit} = ensureCMExtras(cmCache);
    const sel = v.state.selection.main;
    if (sel.empty) {
      runSuperComplete(id, v, '', '', sel.from, sel.to);
      return true;
    }
    const lineFrom = v.state.doc.lineAt(sel.from).from;
    const hl = v.state.doc.sliceString(sel.from, sel.to);
    const dismiss = () => {
      const cur = views.get(id);
      if (!cur || cur.view !== v) return;
      v.dispatch({effects: [setAiEdit.of(null), rec.readOnlyComp.reconfigure(cmCache.state.EditorState.readOnly.of(false))]});
    };
    v.dispatch({
      effects: [
        rec.readOnlyComp.reconfigure(cmCache.state.EditorState.readOnly.of(true)),
        setAiEdit.of({
          lineFrom,
          onCancel() {
            dismiss();
            v.focus();
          },
          onSubmit(instr) {
            v.dispatch({effects: setAiEdit.of(null)});
            runSuperComplete(id, v, hl, instr, sel.from, sel.to);
          },
        }),
      ],
    });
    return true;
  }

## CM6 Extensions Assembly (`buildExtensions`)

Combines language/theme/read-only/line-number compartments, auto-surround brackets, keymaps (`Alt+Enter`, smart expand/shrink, boundary stepping, Tab), and blur/focus listeners.

In [ ]:
#| export_js_iife CRAFT.js craftC2
#| eval: false
function buildExtensions(cm, idOrRef, comps) {
    const ref = typeof idOrRef === 'string' ? {id: idOrRef, key: idOrRef} : idOrRef;
    const {state, view, lang, python, cmds, ac, search} = cm;
    const {ghostField, sigHelpField, setSigHelp, aiEditField, indentGuidesExt, rainbowBracketsExt} = ensureCMExtras(cm);
    const {langComp, themeComp, readOnlyComp} = comps;

    const autoSurroundExt = view.EditorView.inputHandler.of((v, _from, _to, text) => {
      if (v.state.readOnly || text.length !== 1 || !SURROUND_PAIRS[text]) return false;
      const sel = v.state.selection;
      if (!sel.ranges.some(r => !r.empty)) return false;
      const close = SURROUND_PAIRS[text];
      const tr = v.state.changeByRange(r => {
        if (r.empty) return {changes: {from: r.from, to: r.to, insert: text}, range: state.EditorSelection.cursor(r.from + 1)};
        const inner = v.state.doc.sliceString(r.from, r.to);
        return {
          changes: {from: r.from, to: r.to, insert: text + inner + close},
          range: state.EditorSelection.range(r.from + 1, r.to + 1),
        };
      });
      v.dispatch(tr);
      return true;
    });

    const sigKeyHandler = view.EditorView.domEventHandlers({
      keydown(e, v) {
        const cur = v.state.field(sigHelpField, false);
        if (!cur) return false;
        if (['Shift', 'Control', 'Alt', 'Meta'].includes(e.key)) return false;
        if (cur.values.length > 1 && (e.key === 'ArrowUp' || e.key === 'ArrowDown')) {
          e.preventDefault();
          const n = cur.values.length;
          const delta = e.key === 'ArrowDown' ? 1 : -1;
          v.dispatch({effects: setSigHelp.of({...cur, active: (cur.active + delta + n) % n})});
          return true;
        }
        v.dispatch({effects: setSigHelp.of(null)});
        return e.key === 'Escape';
      },
    });

    const prefs = getServerPrefs();
    const smartExpandKey = isMac ? 'Ctrl-Shift-ArrowRight' : 'Alt-Shift-ArrowRight';
    const smartShrinkKey = isMac ? 'Ctrl-Shift-ArrowLeft' : 'Alt-Shift-ArrowLeft';
    const jumpLastKey = isMac ? 'Ctrl-Shift-ArrowUp' : 'Mod-Shift-ArrowUp';
    const acceptOrPickFirst = v => {
      if (ac.completionStatus(v.state) !== 'active') return false;
      if (ac.selectedCompletionIndex(v.state) == null) {
        if (!ac.currentCompletions(v.state).length) return false;
        v.dispatch({effects: ac.setSelectedCompletion(0)});
      }
      return ac.acceptCompletion(v);
    };

    const solveitKeys = state.Prec.high(view.keymap.of([
      {key: 'Shift-Enter', run: () => { saveAndRun(ref.id, 'shift-run'); return true; }},
      {key: 'Mod-Enter', run: () => { saveAndRun(ref.id, 'run'); return true; }},
      {key: 'Alt-Enter', run: () => { saveAndRun(ref.id, 'alt-run'); return true; }},
      {key: 'Mod-Shift-Enter', run: () => { saveAndRun(ref.id, 'continue'); return true; }},
      {key: 'Alt-.', run: v => { requestGhost(ref.id, v, true); return true; }},
      {key: 'ArrowRight', run: v => !getServerPrefs().enterComp && acceptGhost(ref.id, v)},
      {key: 'ArrowUp', run: v => stepBoundary(ref.id, v, true)},
      {key: 'ArrowDown', run: v => stepBoundary(ref.id, v, false)},
      {key: 'Mod-Shift-.', run: v => triggerSuperComplete(ref.id, v)},
      {key: 'Mod-Shift->', run: v => triggerSuperComplete(ref.id, v)},
      {key: 'Ctrl-Shift--', run: v => { splitCell(ref.id, v); return true; }},
      {key: 'Ctrl-Shift-_', run: v => { splitCell(ref.id, v); return true; }},
      {key: 'Mod-Shift-j', run: () => true},
      {key: 'Mod-Shift-k', run: v => { switchCellType(ref.id, v, 'note'); return true; }},
      {key: 'Mod-Shift-l', run: v => { switchCellType(ref.id, v, 'prompt'); return true; }},
      {key: 'Mod-Shift-;', run: v => { switchCellType(ref.id, v, 'raw'); return true; }},
      {key: 'Mod-Shift-d', run: () => { document.getElementById('brain-btn')?.click(); return true; }},
      {key: smartExpandKey, run: v => smartExpandSelection(v)},
      {key: smartShrinkKey, run: v => smartShrinkSelection(v)},
      {key: jumpLastKey, run: () => { jumpToLastCell(ref.id); return true; }},
      {key: 'Alt-Shift-n', run: () => { newCodeCell(); return true; }},
      {key: 'Shift-Escape', run: () => { exitCell(ref.id); return true; }},
      {key: 'Enter', run: v => {
        if (ac.completionStatus(v.state) !== 'active') return false;
        if (ac.selectedCompletionIndex(v.state) != null) return ac.acceptCompletion(v);
        return getServerPrefs().enterComp ? acceptOrPickFirst(v) : false;
      }},
      {key: 'Escape', run: v => {
        if (v.state.field(sigHelpField, false)) { v.dispatch({effects: setSigHelp.of(null)}); return true; }
        if (ac.completionStatus(v.state)) return ac.closeCompletion(v);
        if (v.state.field(ghostField, false)) { cancelGhost(ref.id, v); return true; }
        if (search.searchPanelOpen(v.state)) return search.closeSearchPanel(v);
        if (v.state.selection.ranges.length > 1) return cmds.simplifySelection(v);
        exitCell(ref.id);
        return true;
      }},
      {key: 'Tab', run: v => {
        if (getServerPrefs().enterComp) {
          if (acceptGhost(ref.id, v)) return true;
          return cmds.indentWithTab.run(v);
        }
        if (acceptOrPickFirst(v)) return true;
        return cmds.indentWithTab.run(v);
      }, shift: v => {
        if (v.state.selection.main.empty && fimEnabled() && !v.state.field(sigHelpField, false)) {
          triggerSigHelp(ref.id, v);
          return true;
        }
        return cmds.indentWithTab.shift(v);
      }},
    ]));

    const vscodeKeys = view.keymap.of([
      {key: 'Mod-/', run: cmds.toggleLineComment},
      {key: 'Mod-d', run: search.selectNextOccurrence, preventDefault: true},
      {key: 'Mod-Shift-b', run: search.selectSelectionMatches},
      {key: 'Alt-ArrowUp', run: cmds.moveLineUp},
      {key: 'Alt-ArrowDown', run: cmds.moveLineDown},
      {key: 'Shift-Alt-ArrowUp', run: cmds.copyLineUp},
      {key: 'Shift-Alt-ArrowDown', run: cmds.copyLineDown},
      {key: 'Mod-]', run: cmds.indentMore},
      {key: 'Mod-[', run: cmds.indentLess},
      {key: 'Mod-Shift-[', run: lang.foldCode},
      {key: 'Mod-Shift-]', run: lang.unfoldCode},
      {key: 'Mod-Alt-[', run: lang.foldCode},
      {key: 'Mod-Alt-]', run: lang.unfoldCode},
      {key: 'Mod-f', run: search.openSearchPanel},
      {key: 'Mod-h', run: search.openSearchPanel},
      {key: 'Mod-g', run: search.findNext},
      {key: 'Mod-Shift-g', run: search.findPrevious},
      {key: 'F3', run: search.findNext},
      {key: 'Shift-F3', run: search.findPrevious},
      {key: 'Mod-Space', run: ac.startCompletion},
      {key: 'Mod-i', run: ac.startCompletion},
      ...(prefs.autoClose ? ac.closeBracketsKeymap : []),
      ...ac.completionKeymap.filter(k => k.key !== 'Enter'),
      ...search.searchKeymap,
      ...cmds.historyKeymap,
      ...cmds.defaultKeymap,
      ...lang.foldKeymap,
    ]);

    return [
      solveitKeys,
      vscodeKeys,
      langComp.of(python.python()),
      themeComp.of(themeExtensions(cm)),
      readOnlyComp.of(state.EditorState.readOnly.of(false)),
      ghostField,
      sigHelpField,
      aiEditField,
      prefs.autoClose ? ac.closeBrackets() : autoSurroundExt,
      sigKeyHandler,
      cmds.history(),
      state.EditorState.allowMultipleSelections.of(true),
      view.drawSelection(),
      view.dropCursor(),
      view.rectangularSelection(),
      view.crosshairCursor(),
      view.highlightActiveLine(),
      view.placeholder(`${(kernelLang() || 'python').replace(/^./, c => c.toUpperCase())}…`),
      view.EditorView.lineWrapping,
      lang.indentUnit.of('    '),
      lang.indentOnInput(),
      lang.bracketMatching(),
      lang.codeFolding({placeholderText: '…'}),
      indentGuidesExt,
      rainbowBracketsExt,
      search.search({top: false}),
      search.highlightSelectionMatches(),
      ac.autocompletion({
        override: [kernelComplete, wordComplete],
        activateOnTyping: true,
        defaultKeymap: false,
        selectOnOpen: false,
        interactionDelay: 0,
      }),
      view.tooltips({parent: getTooltipPortal(), position: 'fixed'}),
      view.EditorView.updateListener.of(u => {
        const curId = ref.id;
        states.set(curId, u.state);
        if (u.docChanged) {
          const el = getCellEl(curId);
          if (el) {
            if (u.state.doc.toString() !== getCellText(el)) dirtyIds.add(curId);
            else { dirtyIds.delete(curId); failedSaves.delete(curId); }
            syncSaveState(curId);
          }
          if (u.view.hasFocus && u.state.selection.main.empty) scheduleGhost(curId, u.view);
          else cancelGhost(curId, u.view);
          let line1Changed = false;
          const l1End = u.startState.doc.line(1).to;
          u.changes.iterChangedRanges(fromA => { if (fromA <= l1End) line1Changed = true; });
          if (line1Changed) syncCellLanguage(curId, u.view);
        } else if (u.selectionSet) {
          clearTimeout(ghostTimers.get(curId));
          ghostTimers.delete(curId);
        }
        if (!u.view.hasFocus && ed.activeKey !== curId && !pendingSaves.has(curId) && !dirtyIds.has(curId) && !hasFolds(cm, u.state) && !CORE.optimistic?.isPending?.(curId)) {
          queueMicrotask(() => {
            const k = ref.id, cur = views.get(k);
            if (cur && !cur.view.hasFocus && ed.activeKey !== k && !pendingSaves.has(k) && !dirtyIds.has(k) && !hasFolds(cm, cur.view.state) && !CORE.optimistic?.isPending?.(k)) unmountCell(k);
          });
        }
      }),
    ];
  }

## Cell Mounting, Hotkeys & `window.craftC2` Export

Opens code cells into inline CM6 views (`openCell`), listens for kernel reconnects, registers `Alt+Shift+C` / `Alt+Shift+L` / `Alt+Shift+N` hotkeys, and publishes `window.craftC2`.

In [ ]:
#| export_js_iife CRAFT.js craftC2
#| eval: false
async function openCell(id, selSpec = null) {
    if (ed.destroyed) return;
    const cm = cmCache || await loadCM();
    cmCache = cm;
    if (ed.destroyed) return;
    const el = getCellEl(id), card = getCardIn(el);
    if (!el || !card || hasDiff(el) || card.classList.contains('collapsed')) return;
    if (autoEditGuard && autoEditGuard.id !== id) clearAutoEditGuard();
    const text = getCellText(el);
    ed.openView(id, {
      el, card, text, cm, selSpec,
      hostClass: 'craft-cm-host craft-c2-host',
      extraRec: {langKey: 'python'},
      themeExts: themeExtensions(cm),
      prepareState: ref => {
        const comps = {
          langComp: new cm.state.Compartment(), themeComp: new cm.state.Compartment(),
          readOnlyComp: new cm.state.Compartment(),
        };
        const prev = states.get(id);
        const reusePrev = Boolean(prev && prev.doc.toString() === text && prev._c2Comps?.langComp);
        const edState = reusePrev ? prev : cm.state.EditorState.create({doc: text, extensions: buildExtensions(cm, ref, comps)});
        const activeComps = reusePrev ? prev._c2Comps : comps;
        edState._c2Comps = activeComps;
        return {edState, activeComps, reusePrev};
      },
      onExisting: rec => syncCellLanguage(id, rec.view),
      onMounted: rec => { syncCellLanguage(id, rec.view); },
    });
  }


  const onKernelConnected = () => {
    for (const [id, rec] of views) syncCellLanguage(id, rec.view);
  };

  const ac = new AbortController(), sig = {signal: ac.signal};
  on(document.body, 'kernel-connected', onKernelConnected, sig);

  window.__hotkeyRegister?.({
    id: 'c2-new-code', combo: 'Alt+Shift+N', label: 'New code cell at bottom', group: 'Code',
    handler: () => { newCodeCell(); return true; },
  });

  window.craftC2 = {
    manages: id => isC2El(getCellEl(normId(id))),
    edit: (id, opts) => openCell(normId(id), opts),
    active: () => ed.activeKey,
    view: id => views.get(normId(id ?? ed.activeKey))?.view || null,
    exit: id => exitCell(normId(id ?? ed.activeKey)),
    run: (id, cmd = 'run') => saveAndRun(normId(id ?? ed.activeKey), cmd),
    newCodeCell,
    rekey: (oldId, newId, opts) => ed.rekey(normId(oldId), normId(newId), opts),
    abortOptimistic: id => { const nid = normId(id); dirtyIds.delete(nid); failedSaves.delete(nid); states.delete(nid); unmountCell(nid); },
    expectCodeInsertion: () => queueAutoEdit({forceCode: true}),
    guardNonCodeInsertion(existing) {
      nonCodeInsertion = existing;
      return () => { if (nonCodeInsertion === existing) { nonCodeInsertion = null; schedule(); } };
    },
    dirty: () => [...dirtyIds],
    source: () => cmCache?._source || 'pending',
    disable() {
      clearAutoEditGuard();
      ac.abort();
      for (const t of ghostTimers.values()) clearTimeout(t);
      ghostTimers.clear();
      for (const c of ghostAborts.values()) c.abort();
      ghostAborts.clear();
      for (const c of sigAborts.values()) c.abort();
      sigAborts.clear();
      pendingAutoEdits.length = 0;
      unsubCells();
      ed.disable();
      qsa('.editable.craft-c2, .editable[data-c2-save]').forEach(el => patch(el, {classes: {'craft-c2': false}, dataset: {c2Editing: null, c2Save: null}}));
      qsa('.card-out.craft-c2-traceback').forEach(el => el.classList.remove('craft-c2-traceback'));
      qsa('#craft-c2-tooltips').forEach(el => el.remove());
    },
  };
  sync();

## C2 Inline CodeMirror 6 CSS

In [ ]:
#| export_css CRAFT.css
#| eval: false
/* ── C2 inline CodeMirror 6 cells (base .cm-* & save-dot in 04_cm_core) ── */
.editable:has(> .card-in[data-mtype="code"]):not([data-diff]):not(:has([data-diff])) .card-in[data-mtype="code"] .msg-content pre {
  padding: 4px 15px 4px 4px !important;
  min-height: 18px;
  cursor: text;
}
.editable:has(> .card-in[data-mtype="code"]):not([data-diff]):not(:has([data-diff])) .card-in[data-mtype="code"].craft-c2-mounted .msg-content { display: none !important; }
.craft-c2-host .cm-editor {
  color: var(--craft-fg);
  font-family: var(--craft-code-family) !important;
  font-size: var(--craft-code) !important;
  line-height: var(--craft-code-lh) !important;
  font-weight: 400 !important;
}
.craft-c2-host .cm-scroller { line-height: var(--craft-code-lh) !important; }
.craft-c2-host .cm-content { padding: var(--craft-cm-pad-y, 4px) 15px var(--craft-cm-pad-y, 4px) 0 !important; caret-color: var(--craft-fg); tab-size: 4; min-height: 18px; }
.craft-c2-host .cm-line { padding: 0 0 0 4px !important; line-height: var(--craft-code-lh) !important; }
.craft-c2-host .cm-activeLine, .craft-c2-host .cm-activeLineGutter { background: transparent !important; }
.craft-c2-host .cm-editor.cm-focused .cm-activeLine { background: var(--craft-accent-soft) !important; }
.craft-c2-host .cm-matchingBracket { background: #7aa2f728 !important; outline: 1px solid var(--craft-accent-border); color: inherit !important; }
.craft-c2-host .cm-editor.cm-focused .cm-placeholder { opacity: 0 !important; }
:is(.craft-c2-host, .craft-c2-tooltips) .cm-tooltip {
  background: var(--craft-surface) !important;
  border: 1px solid var(--craft-border) !important;
  border-radius: 6px;
  box-shadow: var(--craft-shadow);
  color: var(--craft-fg);
  font-family: var(--craft-code-family);
  font-size: 11px;
  line-height: 1.35;
  z-index: 1000;
}
:is(.craft-c2-host, .craft-c2-tooltips) .cm-tooltip-autocomplete > ul { max-height: 220px; max-width: min(560px, 90vw); }
:is(.craft-c2-host, .craft-c2-tooltips) .cm-tooltip-autocomplete > ul > li { padding: 2px 8px; display: flex; align-items: baseline; gap: 8px; }
:is(.craft-c2-host, .craft-c2-tooltips) .cm-tooltip-autocomplete > ul > li[aria-selected] { background: var(--craft-accent-bg) !important; color: var(--craft-fg-strong) !important; }
:is(.craft-c2-host, .craft-c2-tooltips) .cm-completionDetail { color: var(--craft-muted); margin-left: auto; font-style: normal; overflow: hidden; text-overflow: ellipsis; white-space: nowrap; max-width: 340px; }
:is(.craft-c2-host, .craft-c2-tooltips) .cm-completionMatchedText { color: var(--craft-accent); text-decoration: none; font-weight: 600; }
.craft-c2-ghost { color: #787c99; opacity: .7; white-space: pre; pointer-events: none; }
.craft-c2-sighelp { max-width: min(560px, 90vw); padding: 6px 8px; }
.craft-c2-sig-head { display: flex; align-items: baseline; gap: 8px; font-weight: 500; }
.craft-c2-sig-label { overflow-wrap: anywhere; }
.craft-c2-sig-active { color: var(--craft-accent); font-weight: 700; text-decoration: underline; }
.craft-c2-sig-nav { margin-left: auto; flex: none; font-size: 10px; color: var(--craft-muted); }
.craft-c2-sig-nav button { border: 0; background: transparent; color: inherit; cursor: pointer; padding: 0 2px; }
.craft-c2-sig-meta { margin-top: 4px; font-size: 10px; color: var(--craft-muted); }
.craft-c2-sig-meta kbd { padding: 0 3px; border-radius: 3px; background: var(--craft-kbd-bg); color: var(--craft-fg); }
.craft-c2-sig-doc { margin: 4px 0 0; max-height: 180px; overflow: auto; white-space: pre-wrap; font-size: 10px; line-height: 1.3; color: var(--craft-fg); }
.craft-c2-ai-form { box-sizing: border-box; padding: 4px; margin: 2px 0; background: var(--craft-recessed); border: 1px solid var(--craft-accent-border); border-radius: 4px; }
.craft-c2-ai-input { height: auto; padding: 3px 6px; font-family: var(--craft-code-family); font-size: 11px; }
.craft-c2-spinner { position: absolute; top: 3px; left: 3px; z-index: 20; pointer-events: none; }

.craft-action-sep { margin: 4px 0; border: 0; border-top: 1px solid #ffffff1a; }
.craft-c2-host .cm-gutters { display: none !important; }
.craft-c2-host .cm-foldPlaceholder { display: inline-block; padding: 0 4px; margin: 0 2px; border: 1px solid var(--craft-accent-border) !important; border-radius: 3px; background: var(--craft-kbd-bg) !important; color: var(--craft-accent) !important; font-size: 9px; line-height: 1; cursor: pointer; }
.craft-c2-host .cm-foldPlaceholder:hover { background: var(--craft-hover) !important; border-color: var(--craft-accent) !important; }

/* Traceback accent, indent guides, rainbow brackets */
.editable:has(> .card-in[data-mtype="code"]):not([data-diff]):not(:has([data-diff])) > .card-in[data-mtype="code"] { position: relative; }
.editable:has(> .card-in[data-mtype="code"]) > .card-out:has(.ansi-red-fg, .ansi-red-intense-fg) { box-shadow: inset 2px 0 0 var(--craft-red) !important; }
.craft-c2-ig { box-shadow: inset 1px 0 0 var(--craft-ig); }
.craft-c2-br-0 { color: var(--craft-amber) !important; }
.craft-c2-br-1 { color: var(--craft-purple) !important; }
.craft-c2-br-2 { color: var(--craft-cyan) !important; }
.craft-c2-br-err { color: var(--craft-red) !important; text-decoration: underline wavy var(--craft-red); }

html.craft-mobile .craft-c2-host { --craft-cm-pad-y: 6px; }
html.craft-mobile :is(.craft-c2-host, .craft-c2-tooltips) .cm-tooltip { font-size: 13px; }
html.craft-mobile :is(.craft-c2-host, .craft-c2-tooltips) .cm-tooltip-autocomplete > ul > li { padding: 6px 10px; }
html.craft-mobile :is(.craft-c2-host, .craft-c2-tooltips) .cm-completionDetail { max-width: 40vw; }
html.craft-mobile .craft-c2-ai-input { font-size: 16px; padding: 8px; }

Verify `10_code_editor_c2.ipynb` exports `window.craftC2`, its templates, and CSS.

In [ ]:
from craft_dev.auth import REPO_ROOT
from craft_dev.build import extract_nb_assets
assets = extract_nb_assets(REPO_ROOT/'nbs/craft/10_code_editor_c2.ipynb')
js = '\n'.join(code for _, code in assets['CRAFT.js'])
for sym in ('window.craftC2', 'c2_tooltip_portal', 'c2_ai_form', 'buildExtensions', 'openCell'):
    assert sym in js, f"Missing {sym}"
assert '.craft-c2-host' in '\n'.join(code for _, code in assets['CRAFT.css'])

## Live `fastcdp` Verification

In [ ]:
#| live
import json
from fastcdp import Rungs
from craft_dev.cdp import open_solveit

async with open_solveit() as (page, _, page_errors):
    rungs = Rungs(page)
    before_ids = await page.eval("[...document.querySelectorAll('#dialog-container > .editable[id]')].map(e => e.id)")
    assert await page.eval("window.craftC2.newCodeCell()") is True
    created_id = None
    try:
        async with rungs('mount and type in temporary C2 cell'):
            created_id = await page.wait_for(
                f"[...document.querySelectorAll('#dialog-container > .editable[id]')].map(e => e.id).find(i => !{json.dumps(before_ids)}.includes(i))",
                timeout=10)
            await page.wait_for_selector(f'#{created_id} .cm-editor .cm-content', timeout=15)
            assert await page.eval("window.craftC2.source() !== 'pending'")
            await page.sel_click(f'#{created_id} .cm-editor .cm-content')
            await page.type('# craft-cdp-test')
            await page.wait_for(f"(document.querySelector('#{created_id} .cm-editor .cm-content')?.innerText || '').includes('# craft-cdp-test')", timeout=3)
            await page.press('Escape')
    finally:
        if created_id:
            await page.eval(f"(() => {{ document.activeElement?.blur?.(); const el = document.getElementById({json.dumps(created_id)}); if (el) window.selectMsg(window.$(el), {{scroll: false}}); }})()")
            await page.press('x')
            await page.wait_for_selector(f'#{created_id}', present=False, timeout=5)
    assert not page_errors, f"Uncaught page errors: {page_errors}"